In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/langchenneto/truncated-reports/truncated_dreams_reports.csv
/kaggle/input/datasets/langchenneto/dreams-emotions1-csv/dreams_emotions1.csv
/kaggle/input/datasets/langchenneto/full-corpus/dream_text.txt


In [2]:
!pip install -q \
    transformers datasets accelerate peft trl bitsandbytes \
    sentence-transformers faiss-cpu chromadb rank-bm25 ragas \
    langchain langchain-community langchain-text-splitters \
    fastapi uvicorn umap-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 16.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 85.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 112.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 24.7 MB/s eta 0:00:00


In [3]:
import os
import json
import time
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F

from transformers import AutoTokenizer
from tqdm import tqdm 

# Viz and statistical work
from sklearn.decomposition import PCA

# Retrieval & Vector Math
import faiss

# NLP & ML
from sentence_transformers import SentenceTransformer, util, CrossEncoder
#from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Setting random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
# create a torch.device object for tensor/model placement
device = torch.device(DEVICE)

Device: cuda
GPU: Tesla T4


In [5]:
# Extract only the text part from dataset and save it in a txt file
DATA_DIR = Path("/kaggle/input")
DATASET_DIR = None
for root, dirs, files in os.walk(DATA_DIR):
    if "dreams_emotions1.csv" in files:
        DATASET_DIR = Path(root)
        break

if DATASET_DIR is None:
    raise FileNotFoundError("dreams_emotions1.csv non trovato in /kaggle/input — controlla che il dataset sia stato aggiunto correttamente")

print(f"Dataset trovato: {DATASET_DIR}")

dreams_path = DATASET_DIR / "dreams_emotions1.csv"

DATASET_DIR_2 = None
for root, dirs, files in os.walk(DATA_DIR):
    if "dream_text.txt" in files:
        DATASET_DIR_2 = Path(root)
        break

if DATASET_DIR_2 is None:
    raise FileNotFoundError("File non trovato in /kaggle/input — controlla che il dataset sia stato aggiunto correttamente")

dream_text_path = DATASET_DIR_2 / "dream_text.txt"

dream_emotions_full = pd.read_csv(dreams_path)
if "dream_text" not in dream_emotions_full.columns:
    raise ValueError("dreams_emotions1.csv must contain a dream_text column")

dream_text = "\n\n".join(
    dream_emotions_full["dream_text"].dropna().astype(str).tolist()
)
target_word_count = int(round(dream_emotions_full["dream_text"].dropna().astype(str).str.split().str.len().mean()))
target_word_count = max(target_word_count, 1)

DATASET_DIR_3 = None
for root, dirs, files in os.walk(DATA_DIR):
    if "truncated_dreams_reports.csv" in files:
        DATASET_DIR_3 = Path(root)
        break

if DATASET_DIR_3 is None:
    raise FileNotFoundError("File non trovato")

Dataset trovato: /kaggle/input/datasets/langchenneto/dreams-emotions1-csv


# **RAG:**

In [6]:
# RAG part -------------------------------------------------------
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device = DEVICE) 

with dream_text_path.open('r', encoding='utf-8') as f:          
    full_text = f.read()

# Configure the LangChain text splitter
# 2000 chars is roughly 350 words
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150,
    chunk_overlap = 30,
    length_function = len,
    separators = ["\n\n", "\n", " ", "", "."]
)

# create the corpus of semantic chunks
corpus = text_splitter.split_text(full_text)

# Embed the chunks
embeddings = embedder.encode(corpus, normalize_embeddings = True)

# FAISS --> Dense Retrieval
dimension = embeddings.shape[1]
# Inner product (requires normalized vectors)
faiss_index = faiss.IndexFlatIP(dimension)
# FAISS requires numpy.float(32) arrays
faiss_vectors = np.array(embeddings).astype("float32")
faiss_index.add(faiss_vectors)

print(f"FAISS Index Total vectors: {faiss_index.ntotal}")

# Search for emotions built from unique macro-emotion labels
query = "anger fear_anxiety sadness_loss joy_amusement love_trust"
emotion_col = None
if "dominant_macro_emotions" in dream_emotions_full.columns:
    emotion_col = "dominant_macro_emotions"
elif "dominant_macro_emotion" in dream_emotions_full.columns:
    emotion_col = "dominant_macro_emotion"

if emotion_col is not None:
    unique_emotions = (
        dream_emotions_full[emotion_col]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
        .unique()
    )
    query = " ".join(unique_emotions.tolist()) or query
    
print(f"\nQuery: {query}")

query_emb = np.array(embedder.encode([query], normalize_embeddings = True)).astype("float32")

# Top k results to fetch
k = 3
distances, indices = faiss_index.search(query_emb, k)

for rank, (idx, score) in enumerate(zip(indices[0], distances[0])):
    print(f"Rank {rank + 1} (Dense Score: {score:.4f}) : {corpus[idx]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Index Total vectors: 187393

Query: love_trust sadness_loss surprise_realization disgust_rejection anger curiosity_desire fear_anxiety joy_amusement shame_remorse optimism_hope
Rank 1 (Dense Score: 0.8567) : Feelings and Thoughts: Happy, angry, sad, scared, worried, grateful. I was worried, angry, sad and scared when my parents told me that we were low on
Rank 2 (Dense Score: 0.8354) : Feelings and Thoughts: Happy, sad, worried. I was worried that Sandy was crying, sad that she was feeling upset, but happy that she finally cried
Rank 3 (Dense Score: 0.8306) : 2. Happy at first, then fear and excitement3. Actual participant4. Unpleasant5. My bedroom6. No7. No


# **LLM**

In [7]:
# LLM part ----------------------------------------------------------
# hyperparameters
batch_size = 16  # how many independent sequences will we process in parallel? 
block_size = 256   # what is the max content length for prediction TO BE CHANGED!!
max_iters = 5000
eval_interval = 500
learning_rate = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200
# Specific for GPT-2 small structural dimensions
n_embd = 512
n_head = 8
n_layer = 6
dropout = 0.1

In [8]:
#------------------------------------------------------
torch.manual_seed(1337)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
# GPT-2 does not have a pad token by default 
tokenizer.pad_token = tokenizer.eos_token

# Update vocab_size
vocab_size = tokenizer.vocab_size

# Train and test splits
data = torch.tensor(tokenizer.encode(full_text), dtype = torch.long)
n = int(0.9*len(data)) # first 90% of the data will be used as train data
train_data = data[:n]
validation_data = data[n:]

# We have to work with chunks of the dataset 
train_data[:block_size +1]  

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (5176387 > 1024). Running this sequence through the model will result in indexing errors


tensor([  464,   530,   379,   262, 21910,    82,   338,  2156,    11,   810,
          340,   338,  5749,  2641,   621,   503,    26,   612,   338,   257,
         3427,  7404,   655,  2641,    11,   351,   257, 22843,  2436, 13631,
         4675,   290,   257,   350,   798,    12,    47,  9346,  3297,   286,
          582,   351, 45731,  4190,    11,   339,   460,   466,  1243,   588,
        24873,   293,   532,   314,   467,   510,   262,   736, 16046,   685,
         8117,  3588,   470,   597,   287,   262,  1103,  2156,    60,   290,
          788,   866,   262,   584,  1735,   685, 20777,   612,   338,   257,
         1218,   900,    11,  3393,    60,   788,   866,   257,  1790,  6565,
        23959,   326,  4962,   257,  5228,    11,   810,   314,  1064,   257,
         7009,  2119,   986,    64,  1862,  2415,   351,  8163,    12, 13664,
        21541,  4190,   287,   257,  2443,  7081,   318,   612,    11, 10801,
          379,   257, 27635,   326,  2048, 23816,   262,  2119, 

In [9]:
def get_batch(split):
    """generate a small batch of data of inputs x and targets y"""
    data = train_data if split == 'train' else validation_data
    # handle short datasets by clamping the sampling range
    max_start = max(1, len(data) - block_size)
    ix = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [10]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [11]:
class Head(nn.Module):

    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

        # Store head_size explicitly for scaling
        self.head_size = head_size
    
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        #mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim =-1)
        wei = self.dropout(wei)
        # perform the weigthed aggregation of the values
        v = self.value(x)
        out = wei @ v  # (B, T, T) @ (B, T, C) --> (B, T, C)
        return out

In [12]:
# Multi-head attention:
class MultiHeadAttention(nn.Module):
    """multiple heads of self attention in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.proj(out)
        return out
    
# To give more time to the logits: 
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)

In [13]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation"""

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x))       
        x = x + self.ffwd(self.ln2(x))
        return x

In [14]:
# Multi-head attention:
class MultiHeadAttention(nn.Module):
    """multiple heads of self attention in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.proj(out)
        return out
    
# To give more time to the logits: 
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)
    
class Block(nn.Module):
    """ Transformer block: communication followed by computation"""

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x))       
        x = x + self.ffwd(self.ln2(x))
        return x

# The simplest one is the Bigram Language Model 
class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(1024, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head = n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)  # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
    
    def forward(self, idx, targets = None):
        B, T = idx.shape
        
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C) = (Batch, Time, Channel) Tensor
        pos_emb = self.position_embedding_table(torch.arange(T, device = device))  # (T, C)
        x = tok_emb + pos_emb  # x holds not only the tokens identity but the position at which they occur
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) # -log-likelihood loss

        return logits, loss
    
    def generate(self, idx, max_new_tokens, temperature = 0.4, top_k = 5, numbers_only = False):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)

            # Apply temperature
            logits = logits / temperature

            # Apply top-k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")

            # apply softmax to get probabilities
            probs = F.softmax(logits, dim = -1)  # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples = 1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat( (idx, idx_next), dim = 1 )  # (B, T+1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)

# the number of parameters
print(sum(p.numel() for p in m.parameters())/1e6, "M parameters")

70.943825 M parameters


In [15]:
# create a Pytorch optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample of a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()

step 0: train loss 11.0056, val loss 11.0022
step 500: train loss 5.1995, val loss 5.1913
step 1000: train loss 4.9491, val loss 4.9621
step 1500: train loss 4.7646, val loss 4.8410
step 2000: train loss 4.6324, val loss 4.7321
step 2500: train loss 4.5031, val loss 4.6320
step 3000: train loss 4.3882, val loss 4.5592
step 3500: train loss 4.3042, val loss 4.4871
step 4000: train loss 4.2135, val loss 4.4159
step 4500: train loss 4.1601, val loss 4.3935


# **BRIDGING**

In [25]:
# BRIDGING RAG WITH LLM -------------------------------------------------------------------------
# Gather the retrieved evidence chunks into a single string
retrieved_texts = [corpus[idx] for idx in indices[0]]  #grab only the first row
evidence_string = "\n".join(retrieved_texts)

# Construct the prompt
prompt = f"Context:\n{evidence_string}\n\nQuestion: {query}\n\nAnswer:"

# Encode the prompt (char-level encoder)
encoded_prompt = tokenizer.encode(prompt, return_tensors = 'pt').to(device)

# Generate the answer: the output will be the original context + new generated tokens
generated_indices = m.generate(encoded_prompt, max_new_tokens = 50)[0].tolist()

# Decode the integers back to text and print
new_tokens_only = generated_indices[encoded_prompt.shape[1]:]
final_output = tokenizer.decode(new_tokens_only, skip_special_tokens = True)
print("----LLM OUTPUT: ----")
print(final_output)

----LLM OUTPUT: ----
30, I think of my dream was going to be able to go to the house. I had to do something to do with my sister, and I was very excited. I was very upset about the dream, and I was very happy about it


**Testing on our dataset**

In [ ]:
test_df = pd.read_csv(DATASET_DIR_3 / "truncated_dreams_reports.csv", sep = ";")

predictions = []

# Evaluation mode
m.eval()

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    query_id = row['Emotion Label']
    query_text = row['Dream Text']
    
    # --- RAG RETRIEVAL ---
    query_emb = np.array(embedder.encode([query_text], normalize_embeddings=True)).astype("float32")
    distances, indices = faiss_index.search(query_emb, k=3)
    
    # Gather the retrieved evidence chunks
    retrieved_texts = [corpus[idx] for idx in indices[0]]
    evidence_string = "\n".join(retrieved_texts)
    
    # --- LLM GENERATION ---
    prompt = f"Similar dreams:\n{evidence_string}\n\nIncomplete dream: {query_text}\n\nContinuation:"
    input_ids = tokenizer.encode(prompt, return_tensors = "pt").to(device)
    input_ids = input_ids[:, -1024:]
    
    # Generate the answer 
    with torch.no_grad(): # Disable gradients for faster inference
        generated_indices = m.generate(input_ids, max_new_tokens=50)[0].tolist()
    
    # Keep only the newly generated tokens
    new_tokens_only = generated_indices[input_ids.shape[1]:]
    final_answer = tokenizer.decode(new_tokens_only, skip_special_tokens= True).strip()
    
    # Save the id and the generated answer
    predictions.append({
        'id': query_id,
        'answer': final_answer
    })

# Create the prediction DataFrame and save it to CSV
results_df = pd.DataFrame(predictions)
results_df.to_csv('predictions.csv', index=False)

print(results_df.head())

 80%|████████  | 33/41 [00:34<00:08,  1.05s/it]